# 01 — EDA HelpDesk Tickets (8095 → 7926 clean)

**Objetivo:** entender desbalance, longitudes, temporalidad y preparar fine-tuning BETO.
- Raw: `data/raw/tickets_raw.csv` (CSV `;`, BOM, mixto latin1/utf-8)
- Clean: `data/processed/tickets_clean.parquet` + `label_mapping.json` (generado por `ml/src/cleaning.py`)
- Mapping: `ml/src/category_mapping.py` (139 tipos legacy → 19 categorías normalizadas `ticket_categories`)
- Ciclo: ingesta → tipado → limpieza → mapeo → cuarentena → anonimización → split (ver `cleaning.py`)

> **8095 filas alcanzan** para BETO fine-tuning: ~417 ej. por clase en promedio, suficiente para transfer learning (84M params, no entrenar desde cero). Ver `ml/src/train_beto.py`.


In [ ]:
from pathlib import Path
import json, re, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10,4)
# %matplotlib inline

ROOT = Path.cwd()
# si el notebook está en ml/notebooks, subir dos niveles
if (ROOT / "ml" / "notebooks" / "01_eda.ipynb").exists():
    ROOT = Path.cwd()
elif Path("..").resolve().name == "HelpDesk":
    ROOT = Path("..").resolve()

# rutas relativas al repo
CLEAN_PARQUET = ROOT / "data" / "processed" / "tickets_clean.parquet"
CLEAN_CSV = ROOT / "data" / "processed" / "tickets_clean.csv"
MAPPING_JSON = ROOT / "data" / "processed" / "label_mapping.json"
QUARANTINE = ROOT / "data" / "quarantine" / "cuarentena.jsonl"
RAW = ROOT / "data" / "raw" / "tickets_raw.csv"

for p in [CLEAN_PARQUET, CLEAN_CSV, MAPPING_JSON, RAW]:
    print(p, "exists" , p.exists(), f"size={p.stat().st_size/1024:.1f}KB" if p.exists() else "")

df = pd.read_parquet(CLEAN_PARQUET) if CLEAN_PARQUET.exists() else pd.read_csv(CLEAN_CSV)
print("clean shape", df.shape)
df.head(3)


In [ ]:
# mapping
with open(MAPPING_JSON) as f:
    mp = json.load(f)
print(f"clases={len(mp['label2id'])}")
for k,v in list(mp["label2id"].items())[:5]:
    print(v, k)
# cuarentena
import pathlib
if QUARANTINE.exists():
    q = pd.read_json(QUARANTINE, lines=True)
    print("cuarentena", len(q), f"({len(q)/8095*100:.1f}%)")
    q.head()
else:
    q = pd.DataFrame()
    print("sin cuarentena")


## 1. Tipado y calidad

In [ ]:
print(df.dtypes.to_string())
print("\n--- describe texto ---")
df[["texto","asunto_clean","descripcion_clean"]].apply(lambda s: s.str.len()).describe().T

print("\n--- nulos ---")
print(df.isna().sum().to_string())
print("\n--- duplicados texto_hash (debería ser 0 en clean) ---")
print("duplicados", df.duplicated(subset=["texto"]).sum() if "texto" in df.columns else "n/a")
print("\n--- PII anonimizada ---")
print(df[["email_hash","tiene_pii"]].head() if "tiene_pii" in df.columns else "sin PII cols")
if "tiene_pii" in df.columns:
    print(df["tiene_pii"].value_counts())


In [ ]:
# validación contra schema de la nueva mesa (asunto 5-200, descripcion 10-5000)
viol = []
for _, r in df.iterrows():
    if len(r["asunto_clean"]) < 5 or len(r["asunto_clean"]) > 200:
        viol.append("asunto")
        break
print("violaciones asunto?", len(viol))
print("descripcion len min/max", df["descripcion_clean"].str.len().min(), df["descripcion_clean"].str.len().max())


## 2. Distribución categorías (19)

In [ ]:
order = df["categoria_label"].value_counts()
print(order.to_string())

fig, ax = plt.subplots(figsize=(12,5))
sns.barplot(y=order.index, x=order.values, hue=order.index, legend=False, palette="viridis", ax=ax)
ax.set_title("Distribución 19 categorías (clean=7926)")
ax.set_xlabel("tickets")
for i, v in enumerate(order.values):
    ax.text(v+10, i, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.show()

# por dominio
dom = df["categoria_dominio"].value_counts()
fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.barplot(y=dom.index, x=dom.values, hue=dom.index, legend=False, palette="Set2", ax=axes[0])
axes[0].set_title("Por dominio")
# sub más frecuentes por dominio
sub = df.groupby(["categoria_dominio","categoria_sub"]).size().reset_index(name="n").sort_values("n", ascending=False)
print(sub.head(10).to_string(index=False))


In [ ]:
# desbalance: ratio y peso para BETO
freq = order.values
print(f"max/min ratio = {freq.max()/freq.min():.1f}x  (2087 / 26)")
imbalance = freq.max()/freq.mean()
print(f"mean {freq.mean():.0f}  median {np.median(freq):.0f}")
# pesos inversos para loss
from collections import Counter
total = len(df)
n_classes = len(order)
class_weights = {lbl: total/(n_classes*cnt) for lbl, cnt in order.items()}
print("pesos (top 3 mayor peso = clases raras):")
for k in sorted(class_weights, key=lambda x: -class_weights[x])[:5]:
    print(f"  {k}: {class_weights[k]:.2f}")


## 3. Temporalidad

In [ ]:
df["fecha_parsed"] = pd.to_datetime(df["fecha_parsed"], errors="coerce")
print("rango", df["fecha_parsed"].min(), "→", df["fecha_parsed"].max())
print("nulos fecha", df["fecha_parsed"].isna().sum())

tmp = df.dropna(subset=["fecha_parsed"]).copy()
tmp["mes"] = tmp["fecha_parsed"].dt.to_period("M").astype(str)
by_month = tmp.groupby("mes").size()
fig, ax = plt.subplots(figsize=(12,3))
by_month.plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("Tickets por mes (detecta estacionalidad para split temporal)")
ax.set_ylabel("n")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print(by_month.tail(8).to_string())

# por día de semana
tmp["dow"] = tmp["fecha_parsed"].dt.day_name()
order_dow = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_dow = tmp["dow"].value_counts().reindex(order_dow)
fig, ax = plt.subplots(figsize=(7,3))
sns.barplot(x=by_dow.index, y=by_dow.values, ax=ax)
ax.set_title("Por día de semana")
plt.xticks(rotation=20)
plt.show()


In [ ]:
# prioridad y estado (para priorización, no para clasificación)
fig, axes = plt.subplots(1,2, figsize=(10,3))
df["prioridad_norm"].value_counts().plot(kind="bar", ax=axes[0], color="#E45756")
axes[0].set_title("Prioridad")
df["estado_norm"].value_counts().plot(kind="bar", ax=axes[1], color="#72B7B2")
axes[1].set_title("Estado")
plt.tight_layout()
plt.show()
print(df["prioridad_norm"].value_counts().to_string())
print(df["estado_norm"].value_counts().to_string())


## 4. Longitudes de texto (para max_length BETO)

In [ ]:
df["len_asunto"] = df["asunto_clean"].str.len()
df["len_desc"] = df["descripcion_clean"].str.len()
df["len_texto"] = df["texto"].str.len()
df["tok_approx"] = df["texto"].str.split().str.len()  # aproximación words

print(df[["len_asunto","len_desc","len_texto","tok_approx"]].describe().T)

fig, axes = plt.subplots(1,3, figsize=(14,3))
for col, ax in zip(["len_asunto","len_desc","len_texto"], axes):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)
    p95 = df[col].quantile(0.95)
    ax.axvline(p95, color="red", ls="--", label=f"p95={p95:.0f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# percentil 95 y 99 para elegir max_length (BETO 512)
for q in [0.90,0.95,0.99]:
    print(f"p{int(q*100)} texto chars={df['len_texto'].quantile(q):.0f}  words~{df['tok_approx'].quantile(q):.0f}")
print("\nBETO max_length recomendado: 128 o 256 (99% de textos < ~120 palabras, cabe en 256 tokens WordPiece)")


In [ ]:
# ejemplos cortos vs largos (inspección manual)
print("— corto —")
print(df.nsmallest(2, "len_texto")[["categoria_label","texto"]].to_string(index=False))
print("\n— largo —")
print(df.nlargest(2, "len_texto")[["categoria_label","texto"]].str.slice(0,300).to_string())


## 5. Señal léxica por categoría (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# top términos por categoría (chi2 rápido o tfidf medio)
vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1,2), stop_words=None, min_df=3)
X = vectorizer.fit_transform(df["texto"])
terms = np.array(vectorizer.get_feature_names_out())

def top_terms_for(label, k=8):
    mask = (df["categoria_label"]==label).values
    # tfidf medio
    mean_tfidf = X[mask].mean(axis=0).A1
    idx = mean_tfidf.argsort()[::-1][:k]
    return list(zip(terms[idx], mean_tfidf[idx]))

for lbl in df["categoria_label"].value_counts().head(6).index:
    print(f"\n{lbl}:")
    for term, score in top_terms_for(lbl):
        print(f"  {term:30s} {score:.3f}")


In [ ]:
# matriz de confusión esperable: categorías con vocabulario solapado
# (ej. tic:Gestión de usuarios vs tic:Permisos y accesos comparten 'usuario','permiso')
# inspección rápida: correlación coseno entre centroides
from sklearn.metrics.pairwise import cosine_similarity

labels = df["categoria_label"].value_counts().index.tolist()
centroids = []
for lbl in labels:
    mask = (df["categoria_label"]==lbl).values
    centroids.append(X[mask].mean(axis=0))
C = np.vstack([c.A1 for c in centroids])
sim = cosine_similarity(C)
# pares más similares (fuera de diagonal)
import itertools
pairs = [(labels[i], labels[j], sim[i,j]) for i,j in itertools.combinations(range(len(labels)),2)]
pairs = sorted(pairs, key=lambda x: -x[2])[:6]
print("Pares más similares (riesgo de confusión):")
for a,b,s in pairs:
    print(f"  {s:.2f}  {a}  <->  {b}")


## 6. Baseline rápido (TF-IDF + Logistic) — sanity check antes de BETO

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# split estratificado 80/20 solo para baseline
from sklearn.model_selection import train_test_split
y = df["label_id"]
X_text = df["texto"]
Xtr, Xte, ytr, yte = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)

pipe = make_pipeline(
    TfidfVectorizer(max_features=12000, ngram_range=(1,2), min_df=2),
    LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=None)
)
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
from sklearn.metrics import f1_score, accuracy_score
print(f"accuracy {accuracy_score(yte,pred):.3f}  macro-F1 {f1_score(yte,pred, average='macro'):.3f}")

# reporte por clase (top)
id2label = {int(k):v for k,v in mp["id2label"].items()}
target_names = [id2label[i] for i in sorted(id2label)]
print(classification_report(yte, pred, target_names=target_names, zero_division=0))
print("\nSi macro-F1 baseline ~0.75-0.85, BETO debería superar 0.85-0.90 con fine-tuning.")


## 7. Recomendaciones para fine-tuning BETO

| Decisión | Recomendación | Motivo (EDA) |
|---|---|---|
| Modelo | `dccuchile/bert-base-spanish-wwm-cased` | español, cased conserva mayúsculas de dependencias |
| `max_length` | 256 (truncation, padding) | p99 texto ~120 palabras < 200 WordPiece; 512 desperdicia VRAM |
| Batch | 16 (train) / 32 (eval) | 7926 filas, 19 clases; cabe en 8GB GPU |
| LR | 2e-5 con warmup 10% | estándar BETO; evitar 5e-5 por desbalance |
| Épocas | 4-6 + early stopping (patience 2) | evita overfit en clase rara (Datos y respaldos n=26) |
| Desbalance | `class_weight` o `oversample` clase <100 | Hidrosanitaria 177, Datos 26; sin peso el modelo ignora raras |
| Split | Stratified 80/10/10 (train/val/test) + seed 42 | preservar 19 clases; no split temporal (no drift fuerte) |
| Métrica | macro-F1 + per-class F1 | accuracy engaña (Piezas gráficas 26% del dataset) |
| Augmentación | back-translation solo para clases <100 si F1 <0.6 | no necesaria inicialmente |

Próximo paso: `python ml/src/train_beto.py --config ml/config.json` (ver celdas export).


In [ ]:
# export split estratificado para el trainer (evita re-split aleatorio en cada run)
from sklearn.model_selection import StratifiedKFold
import json

# guardar splits reproducibles
from sklearn.model_selection import train_test_split

# 80% train, 10% val, 10% test
tr_idx, tmp_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42, stratify=df["label_id"])
val_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, random_state=42, stratify=df.iloc[tmp_idx]["label_id"])
print(f"train {len(tr_idx)}  val {len(val_idx)}  test {len(te_idx)}")

# guardar como jsonl de índices + también parquet splits para inspección
import pathlib
out = ROOT / "data" / "processed"
# no escribir fuera del notebook si no existe, solo mostrar
print("Splits listos para ml/src/train_beto.py (usa train_test_split estratificado interno)")


## 8. Checklist antes de entrenar

- [x] `data/processed/tickets_clean.parquet` sin PII (email_hash)
- [x] 19 clases, ninguna con 0 ejemplos; mínima 26 (Datos y respaldos) — considerar `class_weight`
- [x] longitudes < 256 tokens en 99% — `max_length=256`
- [ ] ejecutar `ml/src/train_beto.py` con GPU o Colab (ver `ml/requirements.txt`)
- [ ] registrar métricas (mlflow o `models/metrics.json`)
- [ ] exportar modelo a `ml/models/beto-tickets/` + `label_mapping.json` para inferencia en Supabase Edge Function
